<div align="center">
  <h1><b> Quantum Communication and Cryptography </b></h1>
  <h2> Quantum Cryptography Protocols </h2>
  <h3> BB84 </h3>
</div>

<br>
<b>Author:</b> <a target="_blank" href="https://github.com/camponogaraviera"> Lucas Camponogara Viera</a>

# Table of Contents

- [Theory](#theory)
    - Introduction
      - Implication of the Non-Orthogonality of the Bases
    - Stages of the Protocol
      - Quantum Transmission Phase 
        - State Preparation (Encoding) by Alice
        - State Measurement by Bob
        - Interception Attempt by Eve
        - Sifting
      - Post-processing
        - Parameter Estimation
        - Error Correction (Reconciliation)
        - Privacy Amplification

- [Qiskit Implementation](#qiskit-implementation)
- [References](#references)

# Theory

## Introduction

BB84 is a four-state, discrete-variable, prepare-and-measure quantum key distribution protocol, and the first scheme proposed by Bennett and Brassard in 1984 for generating secret keys using a random sequence of bits and quantum states in non-orthogonal bases.

BB84 is provably secure under ideal assumptions, assuming a perfect implementation, an authenticated classical channel, and trusted devices/randomness. In this protocol, to generate a cryptographic key, Alice uses two random sequences of classical bits (cbits) to prepare the qubits that will be sent to Bob through a quantum channel. The sequence $\{a\}$ contains random key bits (0 or 1), and the sequence $\{b\}$ contains cbits that represent the bases used to prepare the quantum states of each key bit in $\{a\}$. The two possible bases are non-orthogonal, such as the $Z$ and $X$ bases.

BB84 is a `prepare-and-measure` protocol that `does not use entanglement` and which security fundamentally relies on the no-cloning theorem [no-cloning theorem](./no_cloning.ipynb). The no-cloning theorem states that it is impossible to perfectly clone an arbitrary, unknown quantum state. As a consequence, non-orthogonal quantum states (such as $|0\rangle$ and $|+\rangle$) cannot be perfectly distinguished or cloned. Orthogonal states, such as $|0\rangle$ and $|1\rangle$, are perfectly distinguishable and, therefore, can be cloned if the measurement basis is known. This is why the BB84 protocol is based on quantum state indistinguishability.

### Implication of the Non-Orthogonality of the Bases

The non-orthogonality of the bases ensures that:

1. The non-orthogonality between the two encoding bases in BB84 ensures that an eavesdropper cannot perfectly distinguish the transmitted quantum states and, therefore, cannot perfectly clone them. As a consequence, Eve cannot reliably determine the encoded bit values without introducing disturbance.
   
2. Measurements performed on the wrong basis disturb the state (yielding random outcomes with 50\% probability and collapsing the state), introducing detectable errors in the key. If a qubit prepared in one basis (e.g., X) is measured in a different, incompatible (non-commuting) basis (e.g., Z), the outcome is random, and the post-measurement state is disturbed. If all encoding states were mutually orthogonal, Eve could perform a projective measurement in a common basis and resend an undisturbed copy.

Non-orthogonality prevents perfect state discrimination, which in turn prevents perfect eavesdropping without disturbance.

Non-orthogonality between $|0\rangle$ and $|+\rangle$ means:

$$ \langle 0| +\rangle = \frac{1}{\sqrt{2}} \neq 0.$$

This overlap implies that there is no common measurement basis for both states in which the outcome is deterministic. For example, if the chosen measurement basis is $Z$, then measuring the state $|+\rangle$ will result in the state $|0\rangle$ or $|1\rangle$ with a 50% chance.

- All distinct states within the same orthonormal basis set $\{|o_j\rangle\}_{j=0}^{d-1}$ are mutually orthogonal and normal:

$$ \langle o_i | o_j \rangle = \delta_{ij}. $$

- States from different mutually unbiased bases (MUBs) satisfy:

$$ |\langle o_i | o_j \rangle|^2 = \frac{1}{n}. $$

## Stages of the Protocol

1. Quantum Transmission Phase:
   
  - State preparation: Alice prepares and sends to Bob, through a quantum channel, the qubit states she prepared by randomly choosing between two orthonormal and mutually unbiased bases. The prepared state can be represented by a block of $4n$ qubits (pedagogical choice, not strictly required): $$|\psi\rangle_A = \otimes_{i=1}^{4n} |\psi_{a_ib_i}\rangle_A.$$
  
  - State measurement: Bob randomly chooses one of the two bases used by Alice to measure the received quantum states. Bob receives the following state through the quantum channel $\epsilon$: $$\epsilon |\psi\rangle_A\langle\psi|.$$
  
  - Sifting: Alice reveals to Bob the bases she used, through a public and authenticated public classical channel, without revealing the actual values of her bits.

2. Post-processing (authenticated classical public channel):

  - Parameter Estimation: Sacrifice a random subset of the sifted key (not necessarily "half") to estimate the quantum bit error rate (QBER), i.e., if the error is larger than a threshold, abort the protocol. In practice (with channel noise), the tolerable threshold for secure key extraction is around 11\%. The 25% threshold figure is a common illustrative value for a simple attack.

  - Error Correction (reconciliation): At this stage, a classical error-correction protocol is applied (e.g., Cascade, LDPC) over the authenticated channel to make Alice's and Bob's sifted keys identical, leaking some information to Eve (accounted for later).

  - Privacy Amplification: The remaining cryptographic key is compressed, ensuring that Eve has no residual information.

### Quantum Transmission Phase

#### State Preparation (Encoding) by Alice

Traditionally, Alice prepares the state of each qubit by choosing one of the following two bases:

- Rectilinear (or computational): $∣H\rangle$ or $∣V\rangle$.
- Diagonal (or Hadamard): $∣+\rangle$ or $∣−\rangle$.

Alice's encoding can be represented as follows:

- Cbit A:
    - 0 represents the down state.
    - 1 represents the up state.

- Cbit B:
    - 0 represents the computational/canonical basis (Z-basis).
    - 1 represents the Hadamard basis (X-basis).

Examples:

- If cbit A is 0 and cbit B is 0, Alice prepares the qubit in the state: $|\psi_{00}\rangle = |0\rangle$.
- If cbit A is 1 and cbit B is 0, Alice prepares the qubit in the state: $|\psi_{10}\rangle = |1\rangle$.
- If cbit A is 0 and cbit B is 1, Alice prepares the qubit in the state: $|\psi_{01}\rangle = H|0\rangle = |+\rangle$.
- If cbit A is 1 and cbit B is 1, Alice prepares the qubit in the state: $|\psi_{11}\rangle = H|1\rangle = |-\rangle$.

In the BB84 protocol, the secret key has the same length as the message to be encrypted. This key is also known as a one-time pad because it can only be used once. After its use, it must be discarded.

#### State Measurement by Bob 


Once Bob receives the qubits prepared by Alice, he needs to choose one of the two bases ($Z$ or $X$) to perform a measurement. However, Bob does not know, a priori, in which of the two bases they were prepared. This means that Bob can be wrong 50% of the time when he measures a state that is not an eigenstate of the observable in the chosen basis.

#### Interception Attempt by Eve

Because of the no-cloning theorem, Eve cannot create a copy of the qubits sent by Alice and therefore cannot impersonate Alice. However, Eve can intercept the qubits and perform her own measurements.

Without knowing the bases in which Alice's qubits were prepared, Eve must choose one of the two possible bases to perform a measurement. That is, both Eve and Bob have a $50\%$ chance of choosing the wrong basis for a given qubit. The signature error rate of this simple attack is $1/4 = 25\%$.

In the cases where Eve chooses the wrong basis, that is, when she measures a state that is not an eigenstate of the observable in the chosen basis, the measurement outcome will be probabilistic. This measurement alters the state of the system and causes Eve to be detected.

#### Sifting

Sifting is the final step of the transmission phase, during which the bases chosen by Alice and Bob are compared through an authenticated public classical channel.

At this stage, quantum states whose bases differ must be discarded.

### Post-processing

#### Parameter Estimation

Error estimation (parameters) is the second step in the post-processing process, where the quantum channel bit error rate (quantum bit error rate – QBER) is measured.

Bit errors can arise from several sources: Eavesdropping, noise in the detectors or in the quantum channel, among others.

If the measured QBER is greater than the bit error rate threshold known prior to the presence of the eavesdropper (Eve), the protocol must be reset, since there was interference by the eavesdropper during the transmission of qubits from Alice to Bob.

The QBER measurement is performed by comparing a sufficiently large portion of the measured values.

#### Error Correction (Reconciliation)

The error correction protocol (a.k.a error key reconciliation protocol), is the third step in the post-processing phase used to ensure that Bob and Alice have the same cryptographic key string.

- Error correction codes (general use):

    - **Hamming Code**: A classic linear error-correcting code that detects and corrects single-bit errors using parity bits.

    - **LDPC**: A modern error-correcting code capable of correcting multiple errors using sparse parity-check matrices. More suitable for low-noise channels.

- Protocols for key reconciliation in QKD:

    - **Cascade**: An iterative error-correction protocol for error reconciliation in Quantum Key Distribution (QKD). It identifies and corrects errors using a block-wise approach. It uses binary search within blocks to efficiently locate and correct errors, iteratively refining the process through multiple passes to handle errors missed in previous rounds.

    - **Winnow**: A protocol for efficient error reconciliation in QKD. It uses parity checks on smaller subsets of data (blocks) to correct single-bit errors. Optimized for low-error-rate channels.

Cascade and Winnow were designed specifically for QKD, whereas LDPC and Hamming Code are general-purpose error-correcting codes. Nevertheless, LDPC and Hamming codes can be used to correct single-bit errors in the BB84 protocol. Other protocols use classical neural networks, such as the Tree Parity Machine (TPM), which is based on biologically inspired neural networks.

#### Privacy Amplification

Privacy amplification is the final step used to ensure that Eve has no partial information about the corrected cryptographic key.

The procedure for the amplification step is carried out as follows:

1. A random seed long enough is transmitted through a public channel.
2. This seed is used as a salt in a hash function so that the length of the encrypted key is reduced from $n$ to $m < n$ bits.
   
A common approach to privacy amplification is to use a Toeplitz hash function. The Toeplitz matrix has constant diagonals, and the hash function is based on multiplying the shared key by a random Toeplitz matrix.

# Qiskit Implementation

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer

# &nbsp; <a href="#"><img valign="middle" height="45px" src="https://img.icons8.com/book" width="45" hspace="0px" vspace="0px"></a> References<a name="ref" />

[1] Nielsen, M. A., &#38; Chuang, I. L. (2010). Quantum Computation and Quantum Information: 10th Anniversary Edition. Cambridge: Cambridge University Press. https://doi.org/10.1017/CBO9780511976667

[2] Wolf, R. (2021). Quantum key distribution: An introduction with exercises. Springer. https://doi.org/10.1007/978-3-030-73991-1

[3] Buttler, W. T., Lamoreaux, S. K., Torgerson, J. R., Nickel, G. H., Donahue, C. H., & Peterson, C. G. (2003). Fast, efficient error reconciliation for quantum cryptography. Physical Review A, 67(5), 052303. https://doi.org/10.1103/PhysRevA.67.052303